# 🌱 Crop Recommendation System Using Machine Learning
### A Data Science Based Approach for Crop Selection Using Soil and Environmental Parameters

This notebook demonstrates the end-to-end Data Science workflow:
1. **Unit I**: Data Understanding, Preprocessing & Basic Statistical Description
2. **Exploratory Data Analysis (EDA)** & Visualizations (Matplotlib & Seaborn)
3. **Unit III**: Multi-Class Classification (Decision Tree, KNN, Random Forest)
4. **Model Evaluation & Comparison** (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
5. **Unit IV**: Dimensionality Reduction using Principal Component Analysis (PCA)
6. **Crop Recommendation Inference**

In [ ]:
# Import Required Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style="whitegrid")

## 1. Unit I: Data Loading & Understanding

In [ ]:
# Load dataset
dataset_path = os.path.join("..", "dataset", "crop_data.csv")
if not os.path.exists(dataset_path):
    dataset_path = os.path.join("dataset", "crop_data.csv")

df = pd.read_csv(dataset_path)
print("Dataset Shape:", df.shape)
df.head(10)

In [ ]:
# Dataset Information & Schema
df.info()

In [ ]:
# Statistical Description (df.describe())
df.describe().T

## 2. Data Cleaning & Verification

In [ ]:
# Check for missing and duplicate values
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

## 3. Exploratory Data Analysis & Visualizations

In [ ]:
# Visualization 1: Crop Class Distribution
plt.figure(figsize=(12, 5))
crop_counts = df['label'].value_counts()
sns.barplot(x=crop_counts.index, y=crop_counts.values, palette="crest")
plt.title("Crop Class Frequency Distribution", fontsize=13, fontweight='bold')
plt.xlabel("Crop Name")
plt.ylabel("Sample Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Visualization 2: Correlation Heatmap
numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numerical_cols].corr(), annot=True, fmt=".2f", cmap="YlGnBu", square=True, linewidths=0.5)
plt.title("Correlation Heatmap of Agricultural Parameters", fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Visualization 3: Feature Distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="#2e7d32")
    axes[i].set_title(f"Distribution of {col}", fontweight='bold')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 4: Scatter Plot (Temperature vs Humidity for Sample Crops)
sample_crops = ['rice', 'maize', 'coffee', 'cotton', 'apple', 'banana', 'grapes']
subset_df = df[df['label'].isin(sample_crops)]
plt.figure(figsize=(10, 6))
sns.scatterplot(data=subset_df, x="temperature", y="humidity", hue="label", s=60, palette="tab10")
plt.title("Temperature vs Humidity (Sample Crops)", fontsize=13, fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 4. Unit III: Classification Modeling

In [ ]:
# Separate Features (X) and Target (y)
X = df[numerical_cols]
y_raw = df['label']

# Encode Target Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

# Train/Test Split (80% Train, 20% Test, Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} | Testing set: {X_test.shape[0]}")

In [ ]:
# Initialize the 3 required models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })
    trained[name] = model

comparison_df = pd.DataFrame(results)
display(comparison_df.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1 Score": "{:.2%}"
}))

In [ ]:
# Determine Best Model
best_row = comparison_df.sort_values(by="Accuracy", ascending=False).iloc[0]
best_name = best_row['Model']
print(f"Best Performing Model: {best_name} with Accuracy {best_row['Accuracy']*100:.2f}%")

# Confusion Matrix for Best Model
best_model = trained[best_name]
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGn", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title(f"Confusion Matrix: {best_name}", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Crop")
plt.ylabel("Actual Crop")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf = trained["Random Forest"]
feat_imp = pd.Series(rf.feature_importances_, index=numerical_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=feat_imp.values * 100, y=feat_imp.index, palette="crest_r")
plt.title("Random Forest Feature Importance (%)", fontsize=12, fontweight='bold')
plt.xlabel("Importance (%)")
plt.show()

## 5. Unit IV: Principal Component Analysis (PCA)

In [ ]:
# Standardize Features & Compute PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PC1 Explained Variance: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PC2 Explained Variance: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"Total 2D Variance Captured: {sum(pca.explained_variance_ratio_)*100:.2f}%")

# 2D Scatter Plot
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Crop"] = df["label"]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=pca_df[pca_df["Crop"].isin(sample_crops)], x="PC1", y="PC2", hue="Crop", s=60, palette="tab10")
plt.title(f"2D PCA Projection (Explained Variance: {sum(pca.explained_variance_ratio_)*100:.2f}%)", fontsize=13, fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 6. Crop Recommendation Inference Test

In [ ]:
# Sample User Input: Rice environment conditions
sample_input = np.array([[90, 42, 43, 20.8, 82.0, 6.5, 202.9]])
pred_code = best_model.predict(sample_input)[0]
predicted_crop = label_encoder.inverse_transform([pred_code])[0]

print(f"\n🌱 Input Features: N=90, P=42, K=43, Temp=20.8°C, Humidity=82%, pH=6.5, Rainfall=202.9mm")
print(f"🌱 Predicted Recommended Crop: {predicted_crop.upper()}")